# DWBI Week 4 — Dimensional Modeling
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's Week-3 star (1 fact + 5 dims, 34 rows, 578,650 kg) faces the snowflake question: should DimCity be normalized? Seven tasks count the joins, build the snowflake, do the storage arithmetic, register the relationships, settle cross-filter direction — and the totals do not move by a single kilogram.

## T1 — Hop count audit (12 marks)

**Definition, used literally:** one join operation = one traversal from the fact table to a dimension table
(or between dimension tables). Count them per question, in the star and in the snowflake.

![Star vs snowflake joins](charts/t1_hops.png)

| # | Business question | Dimensions touched | Star joins | Snow joins | Changed? | Why |
|---|---|---|---|---|---|---|
| 1 | Total weight by month | DimDate | 1 | 1 | no | MonthName lives on DimDate in both schemas |
| 2 | Total weight by origin city | DimCity | 1 | 1 | no | CityName lives on DimCity in both schemas |
| 3 | Total weight by province | DimCity | 1 | 2 | **YES** | Province moved out to DimProvince — one extra join |
| 4 | Total weight by country | DimCity | 1 | 3 | **YES** | Country sits two hops out: DimCity → DimProvince → DimCountry |
| 5 | Total cost by province, by month | DimDate, DimCity | 2 | 3 | **YES** | Month unchanged (1); province adds one hop |
| 6 | Total weight by mode and status | DimMode, DimStatus | 2 | 2 | no | Neither dimension was snowflaked |
| 7 | Cost per kg by corridor | DimRoute | 1 | 1 | no | Corridor lives on DimRoute in both schemas |
| 8 | Weight by destination province, by mode | DimCity, DimMode | 2 | 3 | **YES** | Destination city → province is one extra hop; mode unchanged |
| | **TOTALS** | | **11** | **16** | **4** | |

The trap to avoid: questions 1, 2, 6, 7 do **not** change — snowflaking DimCity cannot add joins to
questions that never touch DimCity's snowflaked columns. If your "changed" count is 8, you assumed
every question pays the snowflake tax. Only the four that read Province/Region/Country do.

### The 4-mark sentence

> The eight questions need **5 more joins** in total in the snowflake — and that number tells you
> **nothing about which schema runs faster**. Hop count is not query cost: the extra joins hit a
> 15-row dimension that lives in cache, and real speed depends on data volume, compression, the
> filter path and the engine. **Count, don't assume.**

In [ ]:
# T1 verified in code: 11 star joins, 16 snowflake joins, 4 changed
star = [1,1,1,1,2,2,1,2]; snow = [1,1,2,3,3,2,1,3]
changed = sum(1 for a,b in zip(star,snow) if a!=b)
print(f"star total = {sum(star)} (expect 11)")
print(f"snowflake total = {sum(snow)} (expect 16)")
print(f"changed = {changed} (expect 4); extra joins = {sum(snow)-sum(star)} (expect 5)")
assert sum(star)==11 and sum(snow)==16 and changed==4